# Data cleaning


## Data Loading and initial inspection

- Imported libraries
- Checked data types
- Checked missing values

In [1]:
import pandas as pd
from pathlib import Path

# Load the original dataset
data_path = Path("../data/hotel_bookings.csv")

if not data_path.exists():
    data_path = Path("data/hotel_bookings.csv")

df = pd.read_csv(data_path)

display(df.head())

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
print(f"Dataset shape: {df.shape}")
df.info()

Dataset shape: (119390, 32)
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12 

### Observations: 
- 119,390 rows and 32 features
- Most columns are complete, but company, agent, country, and children contain missing values that need further investigation.
- Children, agent, and company are stored as floats, even though they represent counts or IDs. This is likely due to their missing values.
- reservation_status and reservation_status_date contain information about the final booking outcome and may cause data leakage when predicting is_canceled.

### To fix:
- missing values in company, agent, country, and children
- incorrect data types in children, agent, and company, duplicate and invalid records, unusual adr values, 
- data leakage from reservation_status and reservation_status_date.

## Missing values handling

- missing-value counts and percentages.
- investigate agent, company, country, and children.
- fill missing children values with 0 and country with "Unknown".
- create has_agent and has_company, then remove the original ID columns.


In [3]:
# missing values summary
missing_df = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(4)
})

missing_df = missing_df[
    missing_df["Missing Count"] > 0
].sort_values("Missing Count", ascending=False)

display(missing_df)

,Missing Count,Missing (%)
company,112593,94.3069
agent,16340,13.6862
country,488,0.4087
children,4,0.0034


In [4]:
# overlap between missing agent and company IDs
display(pd.crosstab(
    df["agent"].notna(),
    df["company"].notna(),
    rownames=["Agent Recorded"],
    colnames=["Company Recorded"]
))

# missing country values occur
display(
    df[df["country"].isna()]["hotel"].value_counts()
)

Company Recorded,False,True
Agent Recorded,,
False,9760,6580
True,102833,217


hotel
Resort Hotel    464
City Hotel       24
Name: count, dtype: int64

### Observations 
- children (4 missing): replace with 0, asumme that these bookings didn't have children. 
- country (488 missing): replace with "Unknown" rather than assigning a country or removing the bookings. most missing values (464) from country belong to the Resort Hotel, so deleting these records would disproportionately affect that hotel type
- agent (16,340 missing) and company (112,593 missing): These columns contain IDs rather than numerical measurements. 217 bookings have both IDs recorded and 9,760 have neither. missing IDs do not necessarily mean the booking was made independently, only means that booking agent or company wasn't recorded, so we cannot reliably fill them with specific IDs. Our analysis focuses on booking characteristics rather than individual agencies or companies, we retain whether each ID was recorded using has_agent and has_company, then remove the original ID columns.

In [5]:
# copy for cleaning
df_clean = df.copy()

# handle missing children and country values, fill children with 0 and country with unknown
df_clean["children"] = df_clean["children"].fillna(0).astype(int)
df_clean["country"] = df_clean["country"].fillna("Unknown")

# record whether agent and company IDs are available and convert to has_agent has_company
df_clean["has_agent"] = df_clean["agent"].notna()
df_clean["has_company"] = df_clean["company"].notna()
df_clean = df_clean.drop(columns=["agent", "company"])

In [ ]:
# remaining missing values check
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Dataset shape:", df_clean.shape)

Remaining missing values: 0
Dataset shape: (119390, 32)


## Dupllicate records
- calculated exact duplicates using the original dataset
- compared cancellation rates for repeated and unique records
- show how deduplication would affect the dataset

In [7]:
# duplicates in the original dataset
duplicate_mask = df.duplicated(keep=False)

print("Total bookings:", len(df))
print("Additional duplicate rows:", df.duplicated().sum())
print("All rows involved in duplicates:", duplicate_mask.sum())
print("Unique records:", (~duplicate_mask).sum())

Total bookings: 119390
Additional duplicate rows: 31994
All rows involved in duplicates: 40165
Unique records: 79225


In [8]:
# cancellation rates comparison for repeated and unique records
duplicate_summary = (
    df.assign(
        Record_Type=duplicate_mask.map({
            True: "Repeated",
            False: "Unique"
        })
    )
    .groupby("Record_Type")
    .agg(
        Count=("is_canceled", "size"),
        Cancellation_Rate=("is_canceled", "mean")
    )
)

duplicate_summary["Cancellation_Rate"] = (
    duplicate_summary["Cancellation_Rate"] * 100
).round(2)

display(duplicate_summary)

,Count,Cancellation_Rate
Record_Type,,
Repeated,40165,58.39
Unique,79225,26.22


In [9]:
# compare the dataset with and without exact duplicates
df_deduplicated = df.drop_duplicates()

duplicate_impact = pd.DataFrame({
    "Dataset": ["Original", "Without Duplicates"],
    "Bookings": [len(df), len(df_deduplicated)],
    "Cancellation Rate (%)": [
        df["is_canceled"].mean() * 100,
        df_deduplicated["is_canceled"].mean() * 100
    ]
})

display(duplicate_impact.round(2))

,Dataset,Bookings,Cancellation Rate (%)
0,Original,119390,37.04
1,Without Duplicates,87396,27.49


### Findings
- 31,994 additional identical rows, with 40,165 bookings belonging to repeated groups.
- repeated records had a cancellation rate of 58.39%, compared with 26.22% for unique records.
- removing duplicates would reduce the dataset from 119,390 to 87,396 bookings and decrease the overall cancellation rate from 37.04% to 27.49%.
- duplicates are kept to avoid discarding of valid bookings and substantially changing the cancellation distribution (9 percent)

### Decision
- retain all identical records because there is no unique booking ID to confirm whether they are accidental duplicates or separate reservation
- removing them could discard valid bookings and substantially change the cancellation distribution.


**possibility of duplicated observations is a limitation that should be considered during model evaluation.**

## Invalid Records and Outliers:

In [12]:
# identify potentially invalid or unusual bookings (0 nights, 0 guests, negative adr, zero adr, extremely high adr)
invalid_summary = pd.DataFrame({
    "Issue": [
        "Zero guests",
        "Zero-night stays",
        "Negative ADR",
        "Zero ADR",
        "ADR above 1000"
    ],
    "Count": [
        ((df_clean["adults"] + df_clean["children"] +
          df_clean["babies"]) == 0).sum(),
        ((df_clean["stays_in_week_nights"] +
          df_clean["stays_in_weekend_nights"]) == 0).sum(),
        (df_clean["adr"] < 0).sum(),
        (df_clean["adr"] == 0).sum(),
        (df_clean["adr"] > 1000).sum()
    ]
})

print("Identifying invalid values: ")
display(invalid_summary)

Identifying invalid values: 


,Issue,Count
0,Zero guests,180
1,Zero-night stays,715
2,Negative ADR,1
3,Zero ADR,1959
4,ADR above 1000,1


#### Findings
- 180 bookings with no guests
- 715 with zero-night stays.
- 1 negative ADR
- 1,959 zero-ADR bookings
- 1 very high ADR.

**check whether the invalid conditions overlap to avoid counting the same booking more than once.**

In [18]:
# overlap between bookings with zero guests and zero nights
zero_guests = (
    df_clean["adults"] +
    df_clean["children"] +
    df_clean["babies"] == 0
)

zero_nights = (
    df_clean["stays_in_week_nights"] +
    df_clean["stays_in_weekend_nights"] == 0
)

display(pd.crosstab(
    zero_guests,
    zero_nights,
    rownames=["Zero Guests"],
    colnames=["Zero Nights"],
    margins=False
))

Zero Nights,False,True
Zero Guests,,
False,118565,645
True,110,70


#### Findings:
- 70 have both zero guests and zero nights
- 825 distinct bookings affected by at least one of these conditions
- exclude these records because our analysis focuses on overnight reservations with recorded guests, zero-night bookings could represent day-use reservations rather than errors, but we do not include them. 
- remove the single negative ADR of −€6.38, as it is inconsistent with a standard non-negative room rate

In [19]:
# remove bookings with zero guests, zero nights, or negative ADR
negative_adr = df_clean["adr"] < 0

invalid_mask = zero_guests | zero_nights | negative_adr

print("Records to remove:", invalid_mask.sum())

df_clean = df_clean.loc[~invalid_mask].copy()

print("Remaining records:", len(df_clean))

Records to remove: 826
Remaining records: 118564


In [20]:
# ADR distribution
display(
    df_clean["adr"].describe(
        percentiles=[0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
    ).round(2)
)

# 10 highest room rates
display(
    df_clean.nlargest(10, "adr")[
        ["hotel", "adr", "market_segment",
         "distribution_channel", "customer_type"]
    ]
)

count    118564.00
mean        102.52
std          50.00
min           0.00
25%          70.00
50%          95.00
75%         126.00
95%         194.00
99%         252.00
99.9%       326.40
max        5400.00
Name: adr, dtype: float64

,hotel,adr,market_segment,distribution_channel,customer_type
48515,City Hotel,5400.00,Offline TA/TO,TA/TO,Transient
111403,City Hotel,510.00,Offline TA/TO,TA/TO,Transient
15083,Resort Hotel,508.00,Corporate,Corporate,Transient
103912,City Hotel,451.50,Direct,Direct,Transient-Party
13142,Resort Hotel,450.00,Online TA,TA/TO,Transient
13391,Resort Hotel,437.00,Direct,Direct,Transient
39155,Resort Hotel,426.25,Direct,Direct,Transient
39568,Resort Hotel,402.00,Online TA,TA/TO,Transient
39118,Resort Hotel,397.38,Direct,Direct,Transient
13323,Resort Hotel,392.00,Direct,Direct,Transient


#### Findings:

- highest ADR is €5,400, compared with €510 for the next-highest booking 
- this value is more than ten times the second-highest rate and is clearly separated from the rest of the distribution.

#### Decision:
- remove this single observation because of its extreme rate.

In [21]:
# zero-ADR bookings by market segment
zero_adr = df_clean[df_clean["adr"] == 0]

zero_adr_summary = (
    zero_adr.groupby("market_segment")
    .agg(
        Count=("is_canceled", "size"),
        Cancellation_Rate=("is_canceled", "mean")
    )
    .sort_values("Count", ascending=False)
)

zero_adr_summary["Cancellation_Rate"] = (zero_adr_summary["Cancellation_Rate"] * 100).round(2)

display(zero_adr_summary)

,Count,Cancellation_Rate
market_segment,,
Complementary,652,12.58
Groups,201,9.95
Offline TA/TO,164,18.29
Direct,88,30.68
Online TA,40,10.00
Corporate,20,10.00


#### Findings: 
- 1,165 bookings still have an ADR of zero
- 652 of these belong to the Complementary market segment, they may represent complimentary accommodation. 
- Others are associated with group bookings and different reservation channels.

#### Decision:
- retain these observations because a zero room rate does not necessarily indicate an invalid booking. our goal is to predict cancellations rather than revenue.
- reservations may still provide useful information

In [22]:
# removing the isolated ADR outlier
extreme_adr = df_clean["adr"] > 1000

print("Extreme ADR records removed:", extreme_adr.sum())

df_clean = df_clean.loc[~extreme_adr].copy()

print("Remaining records:", len(df_clean))
print("Maximum ADR:", df_clean["adr"].max())

Extreme ADR records removed: 1
Remaining records: 118563
Maximum ADR: 510.0


In [25]:
# cleaning summary
cleaning_summary = pd.DataFrame({
    "Investigated": [
        "Original dataset",
        "Zero guests / zero-night stays",
        "Negative ADR",
        "Extreme ADR",
        "Final dataset"
    ],
    "Rows Removed": [0, 825, 1, 1, 0],
    "Rows Remaining": [
        len(df),
        len(df) - 825,
        len(df) - 826,
        len(df) - 827,
        len(df_clean)
    ]
})

display(cleaning_summary)

,Investigated,Rows Removed,Rows Remaining
0,Original dataset,0,119390
1,Zero guests / zero-night stays,825,118565
2,Negative ADR,1,118564
3,Extreme ADR,1,118563
4,Final dataset,0,118563


### Observations:
- identify zero-guest bookings, zero-night stays, and unusual ADR values.
- overlap in invalid contidions, removing values that overlap

### Decision:
- a total of 827 observations (0.69%) were removed
-  825 bookings with zero guests or zero nights, one negative ADR, and one extreme ADR of €5,400.
- retained zero-ADR reservations, large group bookings, and other unusual numerical values where there was no clear evidence of an error
- exact duplicate records were also retained

**cleaned dataset now contains 118,563 bookings, down from the original 119,390.**

## Data Consistency

In [27]:
# categorical columns and their values
categorical_cols = df_clean.select_dtypes(include=["object", "string"]).columns

for col in categorical_cols:
    print(f"\n{col}: {df_clean[col].nunique()} categories")
    print(df_clean[col].value_counts(dropna=False).head(20).to_string())


# ranges for the numerical columns
non_negative_cols = [
    "lead_time", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children",
    "babies", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "adr",
    "required_car_parking_spaces",
    "total_of_special_requests"
]

numeric_summary = pd.DataFrame({
    "Minimum": df_clean[non_negative_cols].min(),
    "Maximum": df_clean[non_negative_cols].max(),
    "Negative Values": (df_clean[non_negative_cols] < 0).sum()
})

print("\nNumerical Consistency:")
display(numeric_summary)

# arrival date range 
arrival_dates = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-" +
    df_clean["arrival_date_month"] + "-" +
    df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
    errors="coerce"
)

date_summary = pd.DataFrame({
    "Check": [
        "Invalid arrival dates",
        "Dates before July 2015",
        "Dates after August 2017"
    ],
    "Count": [
        arrival_dates.isna().sum(),
        (arrival_dates < "2015-07-01").sum(),
        (arrival_dates > "2017-08-31").sum()
    ]
})

print("\nDate Consistency:")
display(date_summary)


hotel: 2 categories
hotel
City Hotel      78898
Resort Hotel    39665

arrival_date_month: 12 categories
arrival_date_month
August       13812
July         12586
May          11704
October      11064
April        11045
June         10891
September    10477
March         9714
February      7998
November      6712
December      6689
January       5871

meal: 5 categories
meal
BB           91719
HB           14384
SC           10503
Undefined     1160
FB             797

country: 178 categories
country
PRT    47942
GBR    12094
FRA    10389
ESP     8544
DEU     7285
ITA     3756
IRL     3373
BEL     2338
BRA     2217
NLD     2101
USA     2089
CHE     1724
CN      1278
AUT     1262
SWE     1019
CHN      998
POL      915
ISR      669
RUS      627
NOR      606

market_segment: 8 categories
market_segment
Online TA        56129
Offline TA/TO    24048
Groups           19758
Direct           12449
Corporate         5231
Complementary      715
Aviation           231
Undefined            2

dist

,Minimum,Maximum,Negative Values
lead_time,0.0,709.0,0
stays_in_weekend_nights,0.0,19.0,0
stays_in_week_nights,0.0,50.0,0
adults,0.0,55.0,0
children,0.0,10.0,0
babies,0.0,10.0,0
previous_cancellations,0.0,26.0,0
previous_bookings_not_canceled,0.0,72.0,0
booking_changes,0.0,18.0,0
days_in_waiting_list,0.0,391.0,0



Date Consistency:


,Check,Count
0,Invalid arrival dates,0
1,Dates before July 2015,0
2,Dates after August 2017,0


In [28]:
# Undefined meal plan = Self Catering, equivalent meal categories
df_clean["meal"] = df_clean["meal"].replace("Undefined", "SC")

# final cateogories for meal
display(df_clean["meal"].value_counts())

meal
BB    91719
HB    14384
SC    11663
FB      797
Name: count, dtype: int64

### Findings:
- most categorical values were consistent, except meal, which contained both SC and Undefined, whicih are the same thing.
- market_segment and distribution_channel contained 2 and 5 undefined values
- no negative numerical values remained
- all arrival dates were valid and within July 2015–August 2017.

### Decision:
- combined Undefined and SC in meal, as both represent no meal package.
- kept undefined market segments and distribution channels since their correct values could not be determined.
- Kkept unusual but plausible numerical values.
- no additional rows removed.

## Data Leakage:

In [29]:
# remove columns that directly reveal the booking outcome
leakage_cols = ["reservation_status", "reservation_status_date"]

removed = [col for col in leakage_cols if col in df_clean.columns]
df_clean = df_clean.drop(columns=leakage_cols, errors="ignore")

# features that need to be looked at before modelling
review_cols = [
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "required_car_parking_spaces",
    "total_of_special_requests"
]

print("Columns removed:", removed)
print("Target retained:", "is_canceled" in df_clean.columns)
print("Features requiring timing review:",
      [col for col in review_cols if col in df_clean.columns])
print("Dataset shape:", df_clean.shape)

Columns removed: ['reservation_status', 'reservation_status_date']
Target retained: True
Features requiring timing review: ['assigned_room_type', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests']
Dataset shape: (118563, 30)


### Findings:

- reservation_status directly reveals whether a booking was cancelled or completed.
- reservation_status_date records the final status date, might reveal information unavailable at prediction time.
- assigned_room_type, booking_changes, and other time-sensitive variables may contain information recorded after the original reservation.

### Decision:
- removed reservation_status and reservation_status_date from the cleaned dataset.
- kept is_canceled as the prediction target.
- **kept the other potentially time-sensitive columns for now to be review before modelling.**

## Final checks and export

In [30]:
# final dataset checks
print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)
print("Rows removed:", len(df) - len(df_clean))
print("Remaining missing values:", df_clean.isna().sum().sum())

# cancellation distirbution
cancellation_summary = pd.DataFrame({
    "Count": df_clean["is_canceled"].value_counts(),
    "Percentage (%)": (
        df_clean["is_canceled"].value_counts(normalize=True) * 100
    ).round(2)
})

display(cancellation_summary)

# check sizes match and no missing values and target variable exists
assert df_clean.shape == (118563, 30)
assert df_clean.isna().sum().sum() == 0
assert "is_canceled" in df_clean.columns

#sSave cleaned dataset
output_path = data_path.parent / "hotel_bookings_cleaned.csv"
df_clean.to_csv(output_path, index=False)

# verify saved CSV
df_check = pd.read_csv(output_path)

assert df_check.shape == df_clean.shape
assert df_check.isna().sum().sum() == 0

print(f"\nCleaned dataset saved to: {output_path}")
print("CSV verification passed!")

Original dataset: (119390, 32)
Cleaned dataset: (118563, 30)
Rows removed: 827
Remaining missing values: 0


,Count,Percentage (%)
is_canceled,,
0,74388,62.74
1,44175,37.26



Cleaned dataset saved to: ../data/hotel_bookings_cleaned.csv
CSV verification passed!


### Findings
- cleaned dataset contains 118,563 bookings and 30 columns.
- 827 observations (0.69%) were removed.
- no missing values.
- final cancellation rate is 37.26%, compared with 37.04% in the original dataset

### Decision
- saved the cleaned dataset as hotel_bookings_cleaned.csv
